In [1]:
!pip install pyspark -q

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pyspark.sql import SparkSession
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("NF-ToN-IoT-v2 Model Training") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("SparkSession created successfully")

SparkSession created successfully


In [4]:
PROCESSED_PATH = "/content/drive/MyDrive/BigDataProject/parquet/ton_iot_final"

print("Loading processed data...")
start = time.time()
df_all = spark.read.parquet(PROCESSED_PATH)
print(f"Total rows : {df_all.count():,}")
print(f"Columns    : {df_all.columns}")
print(f"Loaded in  : {time.time() - start:.2f} seconds")

print("\nSplitting data 70/15/15...")
df_train, df_val, df_test = df_all.randomSplit([0.70, 0.15, 0.15], seed=42)

df_train.cache()
df_val.cache()
df_test.cache()

print(f"\nTrain rows      : {df_train.count():,}")
print(f"Validation rows : {df_val.count():,}")
print(f"Test rows       : {df_test.count():,}")
total = df_train.count() + df_val.count() + df_test.count()
print(f"Total           : {total:,}")

Loading processed data...
Total rows : 13,552,396
Columns    : ['features_scaled', 'Attack_Index', 'Label', 'Attack']
Loaded in  : 8.05 seconds

Splitting data 70/15/15...

Train rows      : 9,487,570
Validation rows : 2,030,996
Test rows       : 2,033,830
Total           : 13,552,396


In [5]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

feature_col = "features_scaled"

print(f"Using feature column : {feature_col}")
print(f"Train schema         : {df_train.columns}")

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="Attack_Index",
    predictionCol="prediction",
    metricName="accuracy"
)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="Attack_Index",
    predictionCol="prediction",
    metricName="f1"
)

evaluator_bin_acc = MulticlassClassificationEvaluator(
    labelCol="Label",
    predictionCol="prediction",
    metricName="accuracy"
)

evaluator_bin_f1 = MulticlassClassificationEvaluator(
    labelCol="Label",
    predictionCol="prediction",
    metricName="f1"
)

print("Evaluators ready")

Using feature column : features_scaled
Train schema         : ['features_scaled', 'Attack_Index', 'Label', 'Attack']
Evaluators ready


In [6]:
from pyspark.ml.classification import LogisticRegression

print("Training Logistic Regression...")
start = time.time()

lr = LogisticRegression(
    featuresCol="features_scaled",
    labelCol="Attack_Index",
    maxIter=10,
    regParam=0.01,
    elasticNetParam=0.0,
    family="multinomial"
)

lr_model = lr.fit(df_train)
lr_time  = time.time() - start
print(f"Trained in {lr_time:.2f} seconds")

lr_val_pred = lr_model.transform(df_val)
lr_acc = evaluator_acc.evaluate(lr_val_pred)
lr_f1  = evaluator_f1.evaluate(lr_val_pred)
print(f"Accuracy : {lr_acc:.4f}")
print(f"F1 Score : {lr_f1:.4f}")

Training Logistic Regression...
Trained in 98.69 seconds
Accuracy : 0.6644
F1 Score : 0.6417


In [7]:
from pyspark.ml.classification import DecisionTreeClassifier

print("Training Decision Tree...")
start = time.time()

dt = DecisionTreeClassifier(
    featuresCol="features_scaled",
    labelCol="Attack_Index",
    maxDepth=10,
    seed=42
)

dt_model = dt.fit(df_train)
dt_time  = time.time() - start
print(f"Trained in {dt_time:.2f} seconds")

dt_val_pred = dt_model.transform(df_val)
dt_acc = evaluator_acc.evaluate(dt_val_pred)
dt_f1  = evaluator_f1.evaluate(dt_val_pred)
print(f"Accuracy : {dt_acc:.4f}")
print(f"F1 Score : {dt_f1:.4f}")

Training Decision Tree...
Trained in 81.87 seconds
Accuracy : 0.9565
F1 Score : 0.9560


In [8]:
from pyspark.ml.classification import RandomForestClassifier

print("Training Random Forest...")
start = time.time()

rf = RandomForestClassifier(
    featuresCol="features_scaled",
    labelCol="Attack_Index",
    numTrees=50,
    maxDepth=10,
    seed=42
)
rf_model = rf.fit(df_train)
rf_time  = time.time() - start
print(f"Trained in {rf_time:.2f} seconds")

rf_val_pred = rf_model.transform(df_val)
rf_acc = evaluator_acc.evaluate(rf_val_pred)
rf_f1  = evaluator_f1.evaluate(rf_val_pred)
print(f"Accuracy : {rf_acc:.4f}")
print(f"F1 Score : {rf_f1:.4f}")

Training Random Forest...
Trained in 885.95 seconds
Accuracy : 0.9557
F1 Score : 0.9550


In [9]:
from pyspark.ml.classification import GBTClassifier

print("Training Gradient Boosted Trees...")
start = time.time()

gbt = GBTClassifier(
    featuresCol="features_scaled",
    labelCol="Label",
    maxIter=20,
    maxDepth=5,
    seed=42
)

gbt_model = gbt.fit(df_train)
gbt_time  = time.time() - start
print(f"Trained in {gbt_time:.2f} seconds")

gbt_val_pred = gbt_model.transform(df_val)
gbt_acc = evaluator_bin_acc.evaluate(gbt_val_pred)
gbt_f1  = evaluator_bin_f1.evaluate(gbt_val_pred)
print(f"Accuracy : {gbt_acc:.4f}")
print(f"F1 Score : {gbt_f1:.4f}")

Training Gradient Boosted Trees...
Trained in 600.87 seconds
Accuracy : 0.9809
F1 Score : 0.9808


In [10]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

dt_tuned = DecisionTreeClassifier(
    featuresCol="features_scaled",
    labelCol="Attack_Index",
    seed=42
)

dt_param_grid = ParamGridBuilder() \
    .addGrid(dt_tuned.maxDepth, [5, 10, 15]) \
    .addGrid(dt_tuned.minInstancesPerNode, [1, 5]) \
    .build()

dt_cv = CrossValidator(
    estimator=dt_tuned,
    estimatorParamMaps=dt_param_grid,
    evaluator=evaluator_acc,
    numFolds=3,
    seed=42
)

start = time.time()
dt_cv_model = dt_cv.fit(df_train)
print(f"Tuning completed in {time.time() - start:.2f} seconds")

dt_cv_val_pred = dt_cv_model.transform(df_val)
print(f"Tuned DT Accuracy : {evaluator_acc.evaluate(dt_cv_val_pred):.4f}")
print(f"Tuned DT F1 Score : {evaluator_f1.evaluate(dt_cv_val_pred):.4f}")
print(f"Best maxDepth     : {dt_cv_model.bestModel.depth}")

lr_test_pred  = lr_model.transform(df_test)
dt_test_pred  = dt_cv_model.transform(df_test)
rf_test_pred  = rf_model.transform(df_test)
gbt_test_pred = gbt_model.transform(df_test)

import pandas as pd
results = pd.DataFrame({
    "Model":          ["Logistic Regression", "Decision Tree (Tuned)", "Random Forest", "GBT"],
    "Test Accuracy":  [evaluator_acc.evaluate(lr_test_pred), evaluator_acc.evaluate(dt_test_pred),
                       evaluator_acc.evaluate(rf_test_pred), evaluator_bin_acc.evaluate(gbt_test_pred)],
    "Test F1":        [evaluator_f1.evaluate(lr_test_pred), evaluator_f1.evaluate(dt_test_pred),
                       evaluator_f1.evaluate(rf_test_pred), evaluator_bin_f1.evaluate(gbt_test_pred)],
    "Train Time (s)": [98.69, 81.87, 885.95, 600.87]
})
print(results.to_string(index=False))

gbt_model.save("/content/drive/MyDrive/BigDataProject/models/gbt_model")
dt_cv_model.save("/content/drive/MyDrive/BigDataProject/models/dt_cv_model")
lr_model.save("/content/drive/MyDrive/BigDataProject/models/lr_model")
rf_model.save("/content/drive/MyDrive/BigDataProject/models/rf_model")
print("All models saved")

Tuning completed in 1249.62 seconds
Tuned DT Accuracy : 0.9690
Tuned DT F1 Score : 0.9688
Best maxDepth     : 15
                Model  Test Accuracy  Test F1  Train Time (s)
  Logistic Regression       0.664612 0.642022           98.69
Decision Tree (Tuned)       0.969241 0.969002           81.87
        Random Forest       0.955826 0.955058          885.95
                  GBT       0.980999 0.980940          600.87
All models saved


In [1]:
import pandas as pd, os
OUT = "/content/drive/MyDrive/BigDataProject/tableau_data"
os.makedirs(OUT, exist_ok=True)

pd.DataFrame({
    "Model":["Logistic Regression","Decision Tree","Random Forest","GBT"],
    "Accuracy":[0.6646,0.9692,0.9558,0.9810],
    "F1":[0.6420,0.9690,0.9551,0.9809],
    "Train_Time_s":[98.69,1249.62,885.95,600.87],
    "Task":["Multiclass","Multiclass","Multiclass","Binary"]
}).to_csv(f"{OUT}/model_results.csv", index=False)
print("NB3 done")

NB3 done
